In [12]:
import requests
from datetime import datetime, timedelta
import os
from dotenv import load_dotenv

load_dotenv()

TOKEN_URL = "https://auth.opensky-network.org/auth/realms/opensky-network/protocol/openid-connect/token"
CLIENT_ID = os.getenv("OPENSKY_CLIENT_ID")
CLIENT_SECRET = os.getenv("OPENSKY_CLIENT_SECRET")

# How many seconds before expiry to proactively refresh the token.
TOKEN_REFRESH_MARGIN = 30


class TokenManager:
    def __init__(self):
        self.token = None
        self.expires_at = None

    def get_token(self):
        """Return a valid access token, refreshing automatically if needed."""
        if self.token and self.expires_at and datetime.now() < self.expires_at:
            return self.token
        return self._refresh()

    def _refresh(self):
        """Fetch a new access token from the OpenSky authentication server."""
        r = requests.post(
            TOKEN_URL,
            data={
                "grant_type": "client_credentials",
                "client_id": CLIENT_ID,
                "client_secret": CLIENT_SECRET,
            },
        )
        r.raise_for_status()

        data = r.json()
        self.token = data["access_token"]
        expires_in = data.get("expires_in", 1800)
        self.expires_at = datetime.now() + timedelta(
            seconds=expires_in - TOKEN_REFRESH_MARGIN
        )
        return self.token

    def headers(self):
        """Return request headers with a valid Bearer token."""
        return {"Authorization": f"Bearer {self.get_token()}"}


def get_flight(tokens):
    response = requests.get(
        "https://opensky-network.org/api/states/all",
        headers=tokens.headers(),
    )
    data = response.json()
    return data


# Create a single shared instance for your script.
tokens = TokenManager()

data = get_flight(tokens)

## Response

The response is a JSON object with the following properties:

| Property | Type | Description |
|---|---|---|
| `time` | integer | The time which the state vectors in this response are associated with. All vectors represent the state of a vehicle with the interval [time−1, time]. |
| `states` | array | The state vectors. |

The `states` property is a two-dimensional array. Each row represents a state vector and contains the following fields:

| Index | Property | Type | Description |
|---|---|---|---|
| 0 | `icao24` | string | Unique ICAO 24-bit address of the transponder in hex string representation. |
| 1 | `callsign` | string | Callsign of the vehicle (8 chars). Can be null if no callsign has been received. |
| 2 | `origin_country` | string | Country name inferred from the ICAO 24-bit address. |
| 3 | `time_position` | int | Unix timestamp (seconds) for the last position update. Can be null if no position report was received by OpenSky within the past 15s. |
| 4 | `last_contact` | int | Unix timestamp (seconds) for the last update in general. This field is updated for any new, valid message received from the transponder. |
| 5 | `longitude` | float | WGS-84 longitude in decimal degrees. Can be null. |
| 6 | `latitude` | float | WGS-84 latitude in decimal degrees. Can be null. |
| 7 | `baro_altitude` | float | Barometric altitude in meters. Can be null. |
| 8 | `on_ground` | boolean | Boolean value which indicates if the position was retrieved from a surface position report. |
| 9 | `velocity` | float | Velocity over ground in m/s. Can be null. |
| 10 | `true_track` | float | True track in decimal degrees clockwise from north (north=0°). Can be null. |
| 11 | `vertical_rate` | float | Vertical rate in m/s. Positive = climbing, negative = descending. Can be null. |
| 12 | `sensors` | int[] | IDs of the receivers which contributed to this state vector. Is null if no filtering for sensor was used in the request. |
| 13 | `geo_altitude` | float | Geometric altitude in meters. Can be null. |
| 14 | `squawk` | string | The transponder code aka Squawk. Can be null. |
| 15 | `spi` | boolean | Whether flight status indicates special purpose indicator. |
| 16 | `position_source` | int | Origin of this state's position: `0` = ADS-B, `1` = ASTERIX, `2` = MLAT, `3` = FLARM. |
| 17 | `category` | int | Aircraft category: `0` = No information, `1` = No ADS-B Emitter Category Information, `2` = Light (< 15500 lbs), `3` = Small (15500 to 75000 lbs), `4` = Large (75000 to 300000 lbs), `5` = High Vortex Large (e.g. B-757), `6` = Heavy (> 300000 lbs), `7` = High Performance (> 5g and 400 kts), `8` = Rotorcraft, `9` = Glider / sailplane, `10` = Lighter-than-air, `11` = Parachutist / Skydiver, `12` = Ultralight / hang-glider / paraglider, `13` = Reserved, `14` = Unmanned Aerial Vehicle, `15` = Space / Trans-atmospheric vehicle, `16` = Surface Vehicle – Emergency Vehicle, `17` = Surface Vehicle – Service Vehicle, `18` = Point Obstacle (includes tethered balloons), `19` = Cluster Obstacle, `20` = Line Obstacle. |

In [17]:
for i in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
    print(type(data["states"][0][i]))

<class 'str'>
<class 'str'>
<class 'str'>
<class 'int'>
<class 'int'>
<class 'float'>
<class 'float'>
<class 'float'>
<class 'bool'>
<class 'float'>
<class 'float'>


In [18]:
import pandas as pd
from dataclasses import dataclass


def dict2series(state):
    serie = pd.Series(
        {
            "icao24": state[0],
            "callsign": state[1],
            "origin_country": state[2],
            "time_position": state[3],
            "last_contact": state[4],
            "longitude": state[5],
            "latitude": state[6],
            "baro_altitude": state[7],
            "on_ground": state[8],
            "velocity": state[9],
            "true_track": state[10],
        }
    )
    return serie


@dataclass
class Flight:
    icao24: str
    callsign: str
    origin_country: str
    time_position: int
    last_contact: int
    longitude: float
    latitude: float
    baro_altitude: float
    on_ground: bool
    velocity: float
    true_track: float


def safe_int(val, default=0):
    try:
        return int(val)
    except (ValueError, TypeError):
        return default


def safe_float(val, default=0.0):
    try:
        return float(val)
    except (ValueError, TypeError):
        return default


def safe_bool(val, default=False):
    try:
        return bool(val)
    except (ValueError, TypeError):
        return default


def flight_from_serie(serie):
    return Flight(
        icao24=str(serie["icao24"]),
        callsign=str(serie["callsign"]),
        origin_country=str(serie["origin_country"]),
        time_position=safe_int(serie["time_position"]),
        last_contact=safe_int(serie["last_contact"]),
        longitude=safe_float(serie["longitude"]),
        latitude=safe_float(serie["latitude"]),
        baro_altitude=safe_float(serie["baro_altitude"]),
        on_ground=safe_bool(serie["on_ground"]),
        velocity=safe_float(serie["velocity"]),
        true_track=safe_float(serie["true_track"]),
    )

In [19]:
import dataclasses
import json
from kafka import KafkaProducer


def flight_serializer(flight):
    flight_dict = dataclasses.asdict(flight)
    json_str = json.dumps(flight_dict)
    return json_str.encode("utf-8")


INTERVAL = 90
server = "localhost:9092"
topic_name = "flight-feeds"

producer = KafkaProducer(bootstrap_servers=[server], value_serializer=flight_serializer)

In [20]:
import time

tokens = TokenManager()
MAX_POSITION_AGE = 60

while True:
    data = get_flight(tokens)
    snapshot_ts = data["time"]
    cycle_count = 0
    for state in data["states"]:
        time_position = state[3]
        if time_position is None:
            continue

        if snapshot_ts - time_position > MAX_POSITION_AGE:
            continue

        serie = dict2series(state)
        flight = flight_from_serie(serie)
        producer.send(topic=topic_name, value=flight)
        cycle_count += 1

    producer.flush()
    print(f"{cycle_count} flights sent")
    time.sleep(INTERVAL)

9674 flights sent
9613 flights sent


KeyboardInterrupt: 